# Revision runs (everything that needs the Mac)

Runs the experiments added after the review, using the repository's own
command-line tool, so the results come from exactly the code on GitHub:

1. ISO-NE load, **monthly refits over 2025** (13 rolling windows): naive, trees, deep models, Chronos
2. ISO-NE load with **real archived weather forecasts** at test time
3. ISO-NE price with the **bidding-gate fix** (all models incl. Chronos)
4. Tetouan with the **equal-information features** (deep models and Chronos)
5. EPF markets: robust significance tests and comparison with the **published Lago et al. forecasts**

Every step saves its results and is skipped next time if already done, so you
can stop and restart the notebook at any time. At the end, send
`~/Downloads/revision_results.zip`.

In [ ]:
# ===================================================================
# STEP 1 — SETTINGS
# ===================================================================
import os, sys, subprocess, shutil, time, zipfile
from pathlib import Path
import pandas as pd
from IPython.display import display

HOME = Path.home()
REPO = HOME / "Downloads" / "open-forecast"            # your GitHub clone (updated to v5)
OLD_DATA = HOME / "Downloads" / "open-forecasting-framework-v2" / "data" / "raw"
OLD_RESULTS = HOME / "Downloads" / "open-forecast-complete" / "results"   # earlier full run (EPF forecasts)
EPF_DIRS = [HOME / "Downloads" / "open-forecast-complete" / "data" / "raw" / "epf", HOME / "Downloads"]
DEVICE = "mps"                 # "cpu" if you get MPS errors

RUN_DEEP = True                # deep models in the rolling evaluation (the longest step)
RUN_FORECAST_WEATHER = True    # needs internet (Open-Meteo)
RUN_SEEDS = False              # extra: LSTM with 5 seeds in the rolling evaluation (~3 h)
FULL_DEEP = False              # True: Transformer models in all three sets (several extra hours)

os.environ["OPEN_FORECAST_DEVICE"] = DEVICE
os.environ["PYTORCH_ENABLE_MPS_FALLBACK"] = "1"
os.chdir(REPO)
RES = REPO / "results"
MODES = ["weather_time", "weather_time_history", "weather_time_history_weekly"]
print("Repository:", REPO, "| exists:", REPO.exists())
print("Version check (should say v5):", (REPO / "CHANGELOG.md").read_text().splitlines()[2])


def sh(*args, log=None):
    """Run a command, stream its output here and stop on error."""
    print("$", " ".join(map(str, args)), flush=True)
    start = time.time()
    proc = subprocess.Popen([str(a) for a in args], stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in proc.stdout:
        lines.append(line)
        if not line.lstrip().startswith(("lear: recalibrated", "hist_gradient_boosting: recal", "extra_trees: recal")):
            print(line, end="", flush=True)
    proc.wait()
    if log:
        Path(log).write_text("".join(lines))
    print(f"[{(time.time() - start) / 60:.1f} min, exit code {proc.returncode}]")
    if proc.returncode != 0:
        raise RuntimeError("command failed; see output above")


def cli(*args):
    sh(sys.executable, "-u", "-W", "ignore", "-m", "open_forecast.cli", *args)

In [ ]:
# ===================================================================
# STEP 2 — INSTALL THE PACKAGE AND PUT THE DATA WHERE THE CONFIGS EXPECT IT
# ===================================================================
sh(sys.executable, "-m", "pip", "install", "-q", "-e", ".[all]")

wanted = {REPO / "data/raw/isone" / f"{y}_smd_hourly.xlsx": OLD_DATA / "isone" / f"{y}_smd_hourly.xlsx" for y in (2023, 2024, 2025)}
wanted[REPO / "data/raw/power_consumption.csv"] = OLD_DATA / "power_consumption.csv"
for target, source in wanted.items():
    if not target.exists() and source.exists():
        target.parent.mkdir(parents=True, exist_ok=True); shutil.copy(source, target)
    print("OK     " if target.exists() else "MISSING", target)
for market in ["PJM", "NP", "DE", "BE", "FR"]:
    target = REPO / "data/raw/epf" / f"{market}.csv"
    for d in EPF_DIRS:
        if not target.exists() and (d / f"{market}.csv").exists():
            target.parent.mkdir(parents=True, exist_ok=True); shutil.copy(d / f"{market}.csv", target)
    print("OK     " if target.exists() else "MISSING (only needed for step 8)", target)

In [ ]:
# ===================================================================
# STEP 3 — TESTS (should all pass, including the new leakage audit)
# ===================================================================
sh(sys.executable, "-m", "pytest", "-q")

In [ ]:
# ===================================================================
# STEP 4 — ISO-NE LOAD, MONTHLY REFITS: NAIVE + TREES (13 windows)
# ===================================================================
cli("rolling", "--config", "configs/isone_rolling.yaml",
    "--models", "persistence", "seasonal_naive", "random_forest", "extra_trees", "hist_gradient_boosting",
    "--modes", *MODES, "--output", RES / "isone_rolling_trees")
display(pd.read_csv(RES / "isone_rolling_trees" / "rolling_summary.csv").round(2))

In [ ]:
# ===================================================================
# STEP 5 — ISO-NE LOAD, MONTHLY REFITS: CHRONOS (zero-shot, scored on the same windows)
# ===================================================================
cli("rolling", "--config", "configs/isone_rolling.yaml",
    "--models", "chronos_bolt_small", "chronos2",
    "--modes", "weather_time_history_weekly", "--output", RES / "isone_rolling_foundation")
display(pd.read_csv(RES / "isone_rolling_foundation" / "rolling_summary.csv").round(2))

In [ ]:
# ===================================================================
# STEP 6 — ISO-NE PRICE WITH THE BIDDING-GATE FIX (all models, weekly refits)
# ===================================================================
cli("price", "--config", "configs/isone_price.yaml", "--output", RES / "price_isone_price",
    "--models", "naive", "lear", "hist_gradient_boosting", "extra_trees",
    "chronos_bolt_small", "chronos_bolt_base", "chronos2")
display(pd.read_csv(RES / "price_isone_price" / "price_metrics.csv").round(3))

In [ ]:
# ===================================================================
# STEP 7 — TETOUAN: DEEP MODELS AND CHRONOS WITH THE NEW INFORMATION SETS
# ===================================================================
cli("run", "--config", "configs/tetouan_v2.yaml", "--models", "persistence", "seasonal_naive",
    "chronos_bolt_small", "chronos_bolt_base", "chronos2", "--modes", *MODES,
    "--output", RES / "tetouan_v2_foundation")
if RUN_DEEP:
    cli("run", "--config", "configs/tetouan_v2.yaml", "--models", "persistence", "seasonal_naive",
        "lstm", "cross_attention_transformer", "patchtst", "--modes", *MODES, "--output", RES / "tetouan_v2_deep")
cli("rolling", "--config", "configs/tetouan_rolling.yaml", "--models", "chronos_bolt_small", "chronos2",
    "--modes", "weather_time_history_weekly", "--output", RES / "tetouan_rolling_foundation")
if RUN_DEEP:
    cli("rolling", "--config", "configs/tetouan_rolling.yaml", "--models", "lstm", "cross_attention_transformer",
        "patchtst", "--modes", *MODES, "--output", RES / "tetouan_rolling_deep")

In [ ]:
# ===================================================================
# STEP 8 — EPF MARKETS: ROBUST TESTS + PUBLISHED LAGO ET AL. FORECASTS
# Uses the EPF forecasts you already computed (no re-run). The published
# DNN / LEAR ensemble forecasts are downloaded from the epftoolbox GitHub repo.
# ===================================================================
import numpy as np, urllib.request
from open_forecast.significance import multivariate_dm, holm

PUB = REPO / "data/external/epftoolbox_forecasts"; PUB.mkdir(parents=True, exist_ok=True)
rows = []
for market in ["PJM", "NP", "DE", "BE", "FR"]:
    pub_file = PUB / f"Forecasts_{market}_DNN_LEAR_ensembles.csv"
    if not pub_file.exists():
        urllib.request.urlretrieve("https://raw.githubusercontent.com/jeslago/epftoolbox/master/forecasts/"
                                   f"Forecasts_{market}_DNN_LEAR_ensembles.csv", pub_file)
    pub = pd.read_csv(pub_file, index_col=0, parse_dates=True)
    ours_dir = next(d for d in [RES / f"price_epf_{market.lower()}", OLD_RESULTS / f"price_epf_{market.lower()}"]
                    if (d / "price_predictions.csv").exists() or (d / "price_predictions.csv.gz").exists())
    f = ours_dir / "price_predictions.csv"
    ours = pd.read_csv(f if f.exists() else ours_dir / "price_predictions.csv.gz", parse_dates=["day"])
    ours["hour"] = ours["hour"].astype(str).str.lstrip("h").astype(int)      # "h7" -> 7
    ours["key"] = ours["model"] + "|" + ours["information_set"].fillna("-")
    wide = {k: g.pivot(index="day", columns="hour", values="predicted") for k, g in ours.groupby("key")}
    truth = ours.drop_duplicates(["day", "hour"]).pivot(index="day", columns="hour", values="actual")
    pubd = {c: pub[c].groupby(pub.index.normalize()).apply(lambda s: s.to_numpy()).apply(pd.Series) for c in
            ["LEAR Ensemble", "DNN Ensemble"]}
    days = truth.index.intersection(pubd["DNN Ensemble"].index)
    naive_key = next(k for k in wide if k.startswith("naive|"))
    naive_mae = np.abs(wide[naive_key].loc[days].to_numpy() - truth.loc[days].to_numpy()).mean()
    candidates = {"Chronos-2 + exogenous": wide["chronos2|price_exogenous"], "LEAR (ours) + exogenous": wide["lear|price_exogenous"],
                  "LEAR ensemble (published)": pubd["LEAR Ensemble"], "DNN ensemble (published)": pubd["DNN Ensemble"]}
    t = truth.loc[days].to_numpy()
    for name, frame in candidates.items():
        p = frame.loc[days].to_numpy()
        row = {"market": market, "model": name, "days": len(days), "rMAE": np.abs(p - t).mean() / naive_mae}
        c2 = candidates["Chronos-2 + exogenous"].loc[days].to_numpy()
        if name != "Chronos-2 + exogenous":   # H1: Chronos-2 more accurate than this model
            row["p_iid"] = multivariate_dm(t, p, c2)[1]
            row["p_newey_west"] = multivariate_dm(t, p, c2, lags=7)[1]
            row["p_other_better_nw"] = multivariate_dm(t, c2, p, lags=7)[1]
        rows.append(row)
epf = pd.DataFrame(rows)
mask = epf["p_newey_west"].notna()
epf.loc[mask, "p_holm_nw"] = holm(epf.loc[mask, "p_newey_west"])
(RES / "epf_published_comparison").mkdir(exist_ok=True)
epf.to_csv(RES / "epf_published_comparison" / "comparison.csv", index=False)
display(epf.round(4))

In [ ]:
# ===================================================================
# STEP 9 — FORECAST COMBINATION: CHRONOS-2 + LEAR (equal-weight average)
# No new training: averages the two forecasts already computed, then tests
# the combination against every model and the published Lago et al. ensembles.
# ===================================================================
ens_rows = []
targets = [("ISO-NE", RES / "price_isone_price", None)]
for market in ["PJM", "NP", "DE", "BE", "FR"]:
    dest = RES / f"price_epf_{market.lower()}"
    dest.mkdir(parents=True, exist_ok=True)
    if not (dest / "price_predictions.csv.gz").exists():
        for src in [dest / "price_predictions.csv", OLD_RESULTS / f"price_epf_{market.lower()}" / "price_predictions.csv"]:
            if src.exists():
                pd.read_csv(src).to_csv(dest / "price_predictions.csv.gz", index=False, compression="gzip")
                break
    targets.append((market, dest, PUB / f"Forecasts_{market}_DNN_LEAR_ensembles.csv"))
for market, folder, pub_file in targets:
    args = [sys.executable, "analysis/price_ensemble.py", folder] + ([pub_file] if pub_file else [])
    sh(*args)
    m = pd.read_csv(folder / "ensemble_metrics.csv")
    ens_rows.append(m.assign(market=market))
ens = pd.concat(ens_rows)
display(ens.pivot_table(index="forecast", columns="market", values="rmae").round(3))


In [ ]:
# ===================================================================
# STEP 10 — ISO-NE LOAD WITH REAL WEATHER FORECASTS AT TEST TIME
# Forecasts issued >= 48 h before the valid hour (Open-Meteo, NCEP GFS).
# Models are trained on observed weather, as an operator would.
# ===================================================================
if RUN_FORECAST_WEATHER:
    fc = REPO / "data/raw/isone/weather_forecasts_previous_day2.csv"
    if not fc.exists():
        sh(sys.executable, "scripts/fetch_weather_forecasts.py", "--lead", "2")
    # check: the forecasts must track the observed weather closely
    import open_forecast.isone as iso, yaml
    cfg = yaml.safe_load(open("configs/isone_rolling.yaml"))
    hourly, _ = iso.load_isone_smd(cfg["dataset"]["paths"], cfg["dataset"]["target_columns"])
    forecast = pd.read_csv(fc, parse_dates=["ts"]).set_index("ts")
    joined = hourly[forecast.columns].join(forecast, rsuffix="_fc").loc["2025"]
    check = pd.DataFrame({c: {"corr": joined[c].corr(joined[c + "_fc"]),
                              "MAE_F": (joined[c] - joined[c + "_fc"]).abs().mean()} for c in forecast.columns}).T
    display(check.round(3))
    cli("rolling", "--config", "configs/isone_rolling.yaml", "--weather-forecast", fc,
        "--models", "hist_gradient_boosting", "random_forest", "chronos2", "persistence",
        "--modes", *MODES, "--output", RES / "isone_rolling_forecastweather_trees")
    if RUN_DEEP:
        cli("rolling", "--config", "configs/isone_rolling.yaml", "--weather-forecast", fc,
            "--models", "lstm", "--modes", *MODES, "--output", RES / "isone_rolling_forecastweather_lstm")

In [ ]:
# ===================================================================
# STEP 11 — ISO-NE LOAD, MONTHLY REFITS: DEEP MODELS (longest step; resumable)
# ===================================================================
if RUN_DEEP:
    for model in ["lstm", "cross_attention_transformer", "patchtst"]:
        modes = MODES if (model == "lstm" or FULL_DEEP) else ["weather_time_history_weekly"]
        cli("rolling", "--config", "configs/isone_rolling.yaml", "--models", model,
            "--modes", *modes, "--output", RES / f"isone_rolling_{model}")
        display(pd.read_csv(RES / f"isone_rolling_{model}" / "rolling_summary.csv").round(2))
if RUN_SEEDS:
    cli("rolling", "--config", "configs/isone_rolling.yaml", "--models", "lstm",
        "--modes", "weather_time_history_weekly", "--seeds", "1", "2", "3", "4",
        "--output", RES / "isone_rolling_lstm_seeds")

In [ ]:
# ===================================================================
# STEP 12 — SIGNIFICANCE TESTS AND PAPER TABLES
# ===================================================================
for d in sorted(RES.glob("*rolling*")):
    f = d / "rolling_test_predictions.csv.gz"
    print(d.name, "predictions:", f.exists())
sh(sys.executable, "analysis/paper_tables.py", "results")

In [ ]:
# ===================================================================
# STEP 13 — ZIP THE RESULTS (no raw data, no model weights)
# ===================================================================
zip_path = HOME / "Downloads" / "revision_results.zip"
with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as z:
    for f in RES.rglob("*"):
        in_fold = any(part.startswith("fold_") for part in f.parts)
        if f.is_file() and f.suffix in (".csv", ".gz", ".json", ".txt", ".png", ".pdf") \
                and f.name not in ("predictions.csv", "price_predictions.csv") and "forecasts" not in f.parts \
                and not (in_fold and f.suffix == ".gz"):
            z.write(f, f.relative_to(REPO))
print("Results archive:", zip_path, f"({zip_path.stat().st_size / 1e6:.1f} MB)")